# AIDA 1141 Final Project – 05 Interpretation and the Cleaning-Repairs Check

**Data version 2** (`shared/synthetic-data/`), using notebook 02's saved model and validation predictions and notebook 04's frozen review policy. Nothing is refitted and no threshold or policy changes.

**Question:** What drives the suggestions, in terms the coordinator can use? Why is the simpler rule supported, and when should a person question it? Do the values that cleaning repaired from other records affect any of this?

Notebook 04 froze policy (a): flag an early-warning assessment when the task's start date is 8 or fewer business days away. This notebook explains that choice for the model card and decision brief:

1. **Cleaning repairs (section A).** Cleaning filled some blank employee types and departments from the same employee's other records, and some of those records were created after the assessment. The notebook checks whether the repairs change who the rule flags, then compares the group results and the model's scores with the repaired values, with only the values available at the time, and with no repairs. The procedure is saved so notebooks 06–07 apply it unchanged to the test set and the time-based check.
2. **What the rule captures and misses (section B),** by days until start.
3. **What the comparison model relies on (section C):** its coefficients and permutation importance.
4. **When the coordinator should question the rule (section D):** a description of the at-risk assessments the rule did not flag, and of the flags on assessments that were not at risk.

Everything uses **validation** data, which has already been used for development in notebooks 02–04. The test set and the time-based check stay unopened, and the simulation's evaluation-only values are not used. All records are synthetic: the rates describe the generator's assumptions, not real employees or real HR performance, and the model's associations are not causes.

In [1]:
import hashlib
import importlib.util
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
settings = json.loads((OUTPUT_DIR / "model_settings.json").read_text())          # notebook 02
policy = json.loads((OUTPUT_DIR / "review_policy.json").read_text())             # notebook 04 (frozen)
predictions = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")             # notebook 02
raw_requests = read(SD / "requests/data/requests.csv")                           # export before cleaning
requests = read(SD / "cleaning/output/requests_cleaned.csv")                     # cleaned (version 1.1)
assessments = read(SD / "requests/data/assessments.csv")
events = read(SD / "requests/data/status_events.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

# Every input must be the one the frozen policy was chosen with
FILES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
         "assessments.csv": SD / "requests/data/assessments.csv",
         "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv",
         "model_settings.json": OUTPUT_DIR / "model_settings.json", "model.joblib": OUTPUT_DIR / "model.joblib",
         "validation_predictions.csv": OUTPUT_DIR / "validation_predictions.csv"}
assert {n: sha(p) for n, p in FILES.items()} == policy["fingerprints_sha256"], \
    "An input differs from the fingerprints saved with the frozen policy (notebook 04)"
assert policy["chosen_policy"] == "a" and not policy["model_used"], "The frozen policy is not the days rule"
assert sha(SD / "requests/data/requests.csv") == cleaning_info["source_requests_sha256"] \
    and sha(SD / "requests/data/status_events.csv") == cleaning_info["source_events_sha256"], \
    "The raw export or event log differs from the ones the cleaned file was made from"
model = joblib.load(OUTPUT_DIR / "model.joblib")

# Business-day calendar from the data version 2 generator
spec = importlib.util.spec_from_file_location("generator", SD / "requests/generate_requests.py")
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

# Validation early-warning assessments only (notebook 01's rules); test employees are never loaded into v
split_of = employee_split.set_index("employee_id")["split"]
eligible = assessments[(assessments["employee_id"].map(split_of) == "validation") & (assessments["at_risk"] != "")
                       & (assessments["past_deadline_at_assessment"] == "False")]
assert set(predictions["assessment_id"]) == set(eligible["assessment_id"]) and predictions["assessment_id"].is_unique, \
    "The saved predictions are not exactly the eligible validation assessments"

FEATURES = settings["features"]
v = predictions.merge(assessments[["assessment_id", "assessed_at", "current_status", "business_days_in_current_status",
                                   "team_open_workload"]], on="assessment_id", how="left", validate="one_to_one")
v = v.merge(requests[["request_id", "hiring_department", "intake_channel", "info_issue_at_intake"]],
            on="request_id", how="left", validate="many_to_one")
v["intake_channel"] = v["intake_channel"].replace("", "unknown")              # as in notebook 02
for c in ["business_days_in_current_status", "team_open_workload"]:
    v[c] = v[c].astype(int)
v["assessed_at"] = pd.to_datetime(v["assessed_at"], format="%Y-%m-%dT%H:%M")
v["y"] = v["at_risk"].astype(int)
assert v[FEATURES].notna().all().all() and set(v["employee_id"].map(split_of)) == {"validation"}

DAYS_THRESHOLD = policy["threshold"]
MODEL_THRESHOLD = settings["model_threshold"]
assert np.allclose(model.predict_proba(v[FEATURES])[:, 1], v["risk_probability"], rtol=0, atol=1e-12), \
    "The saved model does not reproduce notebook 02's validation scores"
print(f"Data version {DATA_VERSION}: {len(v):,} validation early-warning assessments, {v['y'].sum()} at risk, "
      f"{v.loc[v['y'] == 1, 'request_id'].nunique()} at-risk tasks, {v['employee_id'].nunique()} employees")
print(f"Frozen policy (a): flag when business days until start <= {DAYS_THRESHOLD}. "
      f"Comparison model threshold (full precision in model_settings.json): {MODEL_THRESHOLD:.4f}")

Data version 2: 1,033 validation early-warning assessments, 194 at risk, 154 at-risk tasks, 209 employees
Frozen policy (a): flag when business days until start <= 8. Comparison model threshold (full precision in model_settings.json): 0.1338


## A. Cleaning repairs: do they change the results?

**What cleaning repaired.** Cleaning version 1.1 filled a blank **employee type** or **hiring department** when the employee's other records all agreed on one value. It also repaired invalid submission times from the request's `submitted` event, but submission time is not a model input or a rule input. The fills were made retrospectively from the full extract, so a filled value can come from a record created **after** the assessment it is used in. Notebook 01 stated this as an assumption; this section tests it.

**Three versions of the two repaired fields,** applied to the same validation assessments; every other input stays the same:

- **Repaired:** the cleaned values used in notebooks 01–04.
- **As of the assessment time:** a filled value is kept only if it could have been known when the assessment was made. Otherwise it becomes unknown.
- **No repairs:** every filled value becomes unknown.

**Donor-availability rule** (for the *as of the assessment time* version). A filled value is available at an assessment only if, at or before the assessment time, at least one of the employee's **other** requests already had a value for that field in the raw export (a value cleaning only standardised, not one it filled):

- **Employee type** is recorded at submission, so a donor's value counts from that donor request's `submitted` event in the status-event log. The event log is used rather than the `submitted_at` field, because cleaning repaired some of those.
- **Hiring department** is confirmed at the information check, so a donor's value counts only from that donor request's information-complete time. A donor still waiting for its information check does not count, which matches cleaning version 1.1.
- If the available donors **disagree**, the value stays unknown. If no donor is available yet, the value is unknown.

The rule is saved to `outputs/repair_check_definition.json` before any comparison is shown, and is protected like the split: notebooks 06–07 must apply it unchanged. Repair counts are reported for requests and for assessments, with the overlap between the two fields.

In [2]:
REPAIRED_FIELDS = ["employee_type", "hiring_department"]
raw = raw_requests.drop_duplicates()                      # cleaning removed exact duplicates first
assert raw["request_id"].is_unique and set(raw["request_id"]) == set(requests["request_id"])
both = requests.merge(raw, on="request_id", suffixes=("", "_raw"), validate="one_to_one")

# A value was filled when the raw export had it blank and the cleaned file does not
filled = pd.DataFrame({f: (both[f + "_raw"].str.strip() == "") & (both[f] != "") for f in REPAIRED_FIELDS})
filled.index = both["request_id"]
for f in REPAIRED_FIELDS:
    logged = next(s["rows"] for s in cleaning_info["steps"] if s["step"] == "employee_level_value_filled" and s["field"] == f)
    assert int(filled[f].sum()) == logged, f"Filled {f} values don't match the cleaning log"

# When each request's own (raw, non-blank) value became available as evidence
submitted_event = pd.to_datetime(events.loc[events["event_type"] == "submitted"].set_index("request_id")["event_at"],
                                 format="%Y-%m-%dT%H:%M")
assert submitted_event.index.is_unique
info_complete = pd.to_datetime(both.set_index("request_id")["info_complete_at"].replace("", None), format="%Y-%m-%dT%H:%M")
assert (both["info_complete_at"] == both["info_complete_at_raw"]).all(), "Cleaning changed an information-complete time"
AVAILABLE_FROM = {"employee_type": submitted_event, "hiring_department": info_complete}
donors = {f: pd.DataFrame({"employee_id": both["employee_id"].to_numpy(), "request_id": both["request_id"].to_numpy(),
                           "value": both[f].to_numpy(),
                           "available_at": both["request_id"].map(AVAILABLE_FROM[f]).to_numpy()})
             .loc[~filled[f].to_numpy() & (both[f].to_numpy() != "")]
          for f in REPAIRED_FIELDS}


def repair_versions(rows):
    # The two repaired fields under each version, for assessment rows with request_id, employee_id and assessed_at
    out = {}
    cleaned = requests.set_index("request_id").loc[rows["request_id"], REPAIRED_FIELDS].set_axis(rows.index)
    for f in REPAIRED_FIELDS:
        was_filled = rows["request_id"].map(filled[f]).astype(bool)
        status = pd.Series("not repaired", index=rows.index)
        as_of = cleaned[f].copy()
        for i in rows.index[was_filled]:
            d = donors[f]
            d = d[(d["employee_id"] == rows.at[i, "employee_id"]) & (d["request_id"] != rows.at[i, "request_id"])
                  & (d["available_at"] <= rows.at[i, "assessed_at"])]
            values = set(d["value"])
            status[i] = ("available" if len(values) == 1 else "conflicting donors" if len(values) > 1
                         else "not yet available")
            if len(values) != 1:
                as_of[i] = "unknown"
            else:
                assert values == {cleaned.at[i, f]}, "An available donor disagrees with the cleaned value"
        out[f] = pd.DataFrame({"repaired": cleaned[f], "as_of_assessment": as_of,
                               "no_repairs": cleaned[f].where(~was_filled, "unknown"), "repair_status": status})
    return out


versions = repair_versions(v)
VERSIONS = ["repaired", "as_of_assessment", "no_repairs"]

# Repair counts: all requests, validation requests in this analysis, and validation assessments, with the overlap between fields
val_requests = filled.loc[filled.index.isin(v["request_id"])]
v_filled = pd.DataFrame({f: v["request_id"].map(filled[f]).astype(bool) for f in REPAIRED_FIELDS})
v_status = pd.DataFrame({f: versions[f]["repair_status"] for f in REPAIRED_FIELDS})
count_rows = []
for level, frame, status in [("requests (all splits)", filled, None),
                             ("validation requests with an early-warning assessment", val_requests, None),
                             ("validation early-warning assessments", v_filled, v_status)]:
    row = {"level": level, "total": len(frame), "employee_type_filled": int(frame["employee_type"].sum()),
           "hiring_department_filled": int(frame["hiring_department"].sum()),
           "both_fields_filled": int(frame.all(axis=1).sum()), "either_field_filled": int(frame.any(axis=1).sum())}
    if status is not None:                                 # timing applies to assessments only
        for f in REPAIRED_FIELDS:
            row[f"{f}_not_yet_available"] = int((status[f] == "not yet available").sum())
            row[f"{f}_conflicting_donors"] = int((status[f] == "conflicting donors").sum())
        row["either_unavailable_as_of_assessment"] = int(status.isin(["not yet available", "conflicting donors"])
                                                         .any(axis=1).sum())
    count_rows.append(row)
repair_counts = pd.DataFrame(count_rows)
for c in repair_counts.columns.drop("level"):
    repair_counts[c] = repair_counts[c].astype("Int64")     # timing columns are blank for the request rows
repair_counts.insert(0, "data_version", DATA_VERSION)
repair_counts.to_csv(OUTPUT_DIR / "repair_counts.csv", index=False)

status_table = pd.concat({f: versions[f].loc[v_filled[f], "repair_status"].value_counts()
                          for f in REPAIRED_FIELDS}, axis=1).fillna(0).astype(int)
status_table.index.name = "donor status (filled values in validation assessments)"

# The donor rule and the three versions, saved and protected before any comparison is shown
definition = {
    "data_version": DATA_VERSION, "cleaning_version": cleaning_info["cleaning_version"],
    "repaired_fields": REPAIRED_FIELDS,
    "filled_value": "raw export blank (after trimming spaces), cleaned value present",
    "versions": {"repaired": "cleaned values, as used in notebooks 01-04",
                 "as_of_assessment": "a filled value is kept only if the donor rule makes it available at the assessment time; otherwise 'unknown'",
                 "no_repairs": "every filled value becomes 'unknown'"},
    "donor_rule": {
        "donors": "the same employee's other requests whose value was present in the raw export (standardised by cleaning, not filled)",
        "employee_type_available_from": "the donor request's 'submitted' event in status_events.csv",
        "hiring_department_available_from": "the donor request's info_complete_at; a donor without one does not count",
        "timing": "available if at or before the assessment time",
        "conflicting_donors": "value becomes 'unknown'",
        "no_available_donor": "value becomes 'unknown'"},
    "unknown_value_in_model": "an unseen category; the saved encoder (handle_unknown='ignore', drop='first') gives it the same indicator values as the reference category",
    "unknown_value_in_groups": "reported as its own 'unknown' group",
    "fixed_items": "same eligible assessments, saved model, full-precision model threshold and frozen days rule in every version",
    "measures": "flags and recall for the days rule; group results by employee type and department; model scores, flags, recall, precision, average precision and ROC AUC",
    "applies_to": "validation here; notebooks 06-07 apply the same rule unchanged to the test set and the time-based check",
}
definition_file = OUTPUT_DIR / "repair_check_definition.json"
if definition_file.exists():
    assert json.loads(definition_file.read_text()) == definition, \
        "The repairs-check definition differs from the saved file; it must not change after it is fixed"
    print("Repairs-check definition matches the saved file")
else:
    definition_file.write_text(json.dumps(definition, indent=2) + "\n")
    print("Repairs-check definition saved")

print("\nRepaired values (counts):")
display(repair_counts.drop(columns="data_version"))
print("Donor availability at the assessment time, for filled values in validation assessments:")
display(status_table)

Repairs-check definition saved

Repaired values (counts):


,level,total,employee_type_filled,hiring_department_filled,both_fields_filled,either_field_filled,employee_type_not_yet_available,employee_type_conflicting_donors,hiring_department_not_yet_available,hiring_department_conflicting_donors,either_unavailable_as_of_assessment
0,requests (all splits),4364,44,44,1,87,<NA>,<NA>,<NA>,<NA>,<NA>
1,validation requests with an early-warning asse...,839,12,9,0,21,<NA>,<NA>,<NA>,<NA>,<NA>
2,validation early-warning assessments,1033,16,12,0,28,4,0,4,0,8


Donor availability at the assessment time, for filled values in validation assessments:


,employee_type,hiring_department
donor status (filled values in validation assessments),,
available,12,8
not yet available,4,4


**Does any repair reach the days rule?** The rule uses only business days until start, which comes from the assessment time and the confirmed start date. The cell below checks those inputs rather than assuming they are untouched:

- **Start date:** the confirmed start date in the raw export, compared with the cleaned file. A raw date that is missing or can't be parsed is counted separately from one that parses but differs; neither is counted as unchanged.
- **Assessment time:** the assessments file is not processed by cleaning. Comparing the analysis table with that file only confirms that the timestamp was preserved through the merge; it is not an independent check. At checkpoint 1, the assessment time is also compared with the request's raw information-complete time, which is an independent cross-check.
- **Days until start** is recalculated from the raw start date with the generator's calendar and compared with the saved value, and the frozen rule's flags are compared across the three versions, assessment by assessment.

If any count of discrepancies is not zero, the notebook stops before the comparisons that follow.

In [3]:
start_raw = v["request_id"].map(raw.set_index("request_id")["confirmed_start_date"])
start_clean = v["request_id"].map(requests.set_index("request_id")["confirmed_start_date"])
parsed_raw = pd.to_datetime(start_raw.replace("", None), format="%Y-%m-%d", errors="coerce")
unparseable = parsed_raw.isna()                           # missing, unmatched or not a valid date
differs = parsed_raw.notna() & (start_raw != start_clean)

source_time = v["assessment_id"].map(assessments.set_index("assessment_id")["assessed_at"])
cp1 = v["checkpoint"] == "checkpoint_1"
days_from_raw = pd.Series([generator.business_days_until_start(a.date(), s.date()) if pd.notna(s) else np.nan
                           for a, s in zip(v["assessed_at"], parsed_raw)], index=v.index)
# Each version's full input table: only the two repaired fields differ. The frozen rule's saved expression
# is applied to each table, so any dependence on a repaired field would show up as a changed flag.
inputs = {ver: v[FEATURES].assign(business_days_until_start=days_from_raw,
                                   **{f: versions[f][ver] for f in REPAIRED_FIELDS}) for ver in VERSIONS}
rule_flags = {ver: inputs[ver].eval(policy["flag_rule"]) for ver in VERSIONS}
rule_audit = pd.Series({
    "validation early-warning assessments": len(v),
    "raw confirmed start date missing or unparseable": int(unparseable.sum()),
    "raw confirmed start date parseable but different from the cleaned value": int(differs.sum()),
    "assessment timestamp not preserved from the source file (merge check)":
        int((v["assessed_at"].dt.strftime("%Y-%m-%dT%H:%M") != source_time).sum()),
    "checkpoint 1 time differs from the raw information-complete time (independent cross-check)":
        int((v.loc[cp1, "assessed_at"] != v.loc[cp1, "request_id"].map(info_complete)).sum()),
    "days until start recalculated from the raw start date differs from the saved value":
        int((days_from_raw != v["business_days_until_start"]).sum()),
    "rule flag differs from notebook 02's saved flag": int((rule_flags["repaired"] != v["days_rule"]).sum()),
    "rule flag differs between the repaired and as-of-assessment versions":
        int((rule_flags["repaired"] != rule_flags["as_of_assessment"]).sum()),
    "rule flag differs between the repaired and no-repairs versions":
        int((rule_flags["repaired"] != rule_flags["no_repairs"]).sum()),
}, name="assessments")
rule_audit.index.name = "rule input check"
rule_audit.to_frame().reset_index().assign(data_version=DATA_VERSION).to_csv(OUTPUT_DIR / "repair_rule_audit.csv",
                                                                             index=False)
display(rule_audit.to_frame())
# Stop here if any check found a discrepancy: a missing date would otherwise look like a valid 'not flagged'
assert (rule_audit.drop("validation early-warning assessments") == 0).all(), \
    "The rule-input audit found a discrepancy; the repair comparison cannot continue"

,assessments
rule input check,
validation early-warning assessments,1033
raw confirmed start date missing or unparseable,0
raw confirmed start date parseable but different from the cleaned value,0
assessment timestamp not preserved from the source file (merge check),0
checkpoint 1 time differs from the raw information-complete time (independent cross-check),0
days until start recalculated from the raw start date differs from the saved value,0
rule flag differs from notebook 02's saved flag,0
rule flag differs between the repaired and as-of-assessment versions,0
rule flag differs between the repaired and no-repairs versions,0
